# Seeing with Transformers: ViT, CLIP and Vision-Language Models

The runnable code of [Seeing with Transformers: ViT, CLIP and Vision-Language Models](https://residual-stream.lvtan71.workers.dev/posts/vision-transformers-and-multimodal/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/vision-transformers-and-multimodal.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 2 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "vision-transformers-and-multimodal"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['data/clip-diagram-renders.tar.gz', 'data/blog-11f17f9.tar.gz']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## Patches: an image as a sequence

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/vision-transformers-and-multimodal/#patches-an-image-as-a-sequence)

In [ ]:
# An image as a sequence. ViT-B/16 on a 224x224 RGB image: cut it into
# 16x16 patches, flatten each (16 * 16 * 3 numbers), and project each to the
# model's width with one shared linear layer, the patch embedding.
H = W = 224; P = 16; C = 3; D = 768
n = (H // P) * (W // P)
print(f"{H}x{W} image, {P}x{P} patches: {H // P} x {W // P} = {n} patches, each {P * P * C} numbers")
print(f"patch embedding: a {P * P * C} x {D} matrix + bias = {P * P * C * D + D:,} parameters")
print(f"sequence for the Transformer: {n} patch tokens + 1 [CLS] = {n + 1} positions, attention over {(n + 1) ** 2:,} pairs")
for res, p in ((224, 14), (336, 14), (448, 14), (1024, 16)):
    print(f"  {res}x{res} with {p}x{p} patches: {(res // p) ** 2:,} tokens")

# In a vision-language model the patch embeddings, after a projector, become
# tokens in the language model's context, and cost KV cache like text.
# LLaVA-1.5: a CLIP ViT-L/14 at 336x336 gives 576 image tokens, fed to a 7B
# language model (32 layers, width 4096, full multi-head attention, 16-bit).
kv_per_token = 32 * 2 * 4096 * 2                 # layers x (K and V) x width x bytes
print(f"\nLLaVA-1.5 (7B language model): {kv_per_token / 2 ** 10:.0f} KiB of KV cache per token")
print(f"  one image, 576 tokens: {576 * kv_per_token / 2 ** 20:.0f} MiB; a 20-image conversation: "
      f"{20 * 576 * kv_per_token / 2 ** 30:.1f} GiB")
print(f"  as text, 576 tokens is about {int(576 * 0.75)} English words")

The post printed:

```text
224x224 image, 16x16 patches: 14 x 14 = 196 patches, each 768 numbers
patch embedding: a 768 x 768 matrix + bias = 590,592 parameters
sequence for the Transformer: 196 patch tokens + 1 [CLS] = 197 positions, attention over 38,809 pairs
  224x224 with 14x14 patches: 256 tokens
  336x336 with 14x14 patches: 576 tokens
  448x448 with 14x14 patches: 1,024 tokens
  1024x1024 with 16x16 patches: 4,096 tokens

LLaVA-1.5 (7B language model): 512 KiB of KV cache per token
  one image, 576 tokens: 288 MiB; a 20-image conversation: 5.6 GiB
  as text, 576 tokens is about 432 English words
```

## CLIP: images and text in one space

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/vision-transformers-and-multimodal/#clip-images-and-text-in-one-space)

In [ ]:
import numpy as np

# CLIP's loss on a batch of 3 image-caption pairs. Embeddings are unit
# vectors; S[i, j] is the cosine similarity of image i and caption j, and the
# logits are S divided by a temperature. The matching pairs are on the diagonal.
S = np.array([[0.90, 0.20, 0.10],
              [0.30, 0.80, 0.40],
              [0.05, 0.35, 0.70]])
t = 0.07                                   # CLIP's initial temperature
L = S / t
def log_softmax(x, axis):
    x = x - x.max(axis=axis, keepdims=True)
    return x - np.log(np.exp(x).sum(axis=axis, keepdims=True))
img_to_txt = -np.diag(log_softmax(L, axis=1))     # each image: which caption is mine? (cross-entropy over the row)
txt_to_img = -np.diag(log_softmax(L, axis=0))     # each caption: which image is mine? (over the column)
print("logits = similarities / 0.07:\n", L.round(2))
print("image -> caption losses:", img_to_txt.round(4), " caption -> image losses:", txt_to_img.round(4))
print(f"CLIP loss (symmetric InfoNCE) = mean of both directions = {(img_to_txt.mean() + txt_to_img.mean()) / 2:.4f}")

# SigLIP: no softmax over the batch. Every pair is its own binary question,
# match (label +1) or not (-1), scored with a sigmoid; a learned scale t
# (from log t, initialized at log 10) and bias b (initialized at -10).
tp, b = 10.0, -10.0
z = np.where(np.eye(3) == 1, 1, -1)
pair = np.log1p(np.exp(-z * (tp * S + b)))        # -log sigmoid(z * logit)
print("\nSigLIP per-pair losses:\n", pair.round(4))
print(f"SigLIP loss = sum over all 9 pairs / 3 images = {pair.sum() / 3:.4f}")

The post printed:

```text
logits = similarities / 0.07:
 [[12.86  2.86  1.43]
 [ 4.29 11.43  5.71]
 [ 0.71  5.   10.  ]]
image -> caption losses: [0.0001 0.0041 0.0068]  caption -> image losses: [0.0002 0.0018 0.0139]
CLIP loss (symmetric InfoNCE) = mean of both directions = 0.0045

SigLIP per-pair losses:
 [[1.3133e+00 3.0000e-04 1.0000e-04]
 [9.0000e-04 2.1269e+00 2.5000e-03]
 [1.0000e-04 1.5000e-03 3.0486e+00]]
SigLIP loss = sum over all 9 pairs / 3 images = 2.1647
```

## CLIP on this blog's diagrams

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/vision-transformers-and-multimodal/#clip-on-this-blogs-diagrams)

**Not shown in the post:** the post rendered each diagram with headless Chrome on a Mac (the cell above it in the post). Those renders are in `fit/` and `notext/`, unpacked from `clip-diagram-renders.tar.gz` by the setup cell; this checks they are there.

In [ ]:
import glob
print(len(glob.glob("fit/*.png")), len(glob.glob("notext/*.png")))   # the post's render step printed 19 19

In [ ]:
import glob
import re
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

# CLIP (ViT-B/32) on this blog's own diagrams. Each diagram's caption is the
# alt text its post gives it. Can CLIP find the right diagram from its
# caption, and the right caption from the diagram, among the 19? The
# diagrams are rendered whole into 512x512 squares (render.py), once as
# they are and once with every text label hidden.
alts = {}
for path in glob.glob("content/posts/*.mdx"):
    for m in re.finditer(r'light="/images/diagrams/([^"]+)-light\.svg"\s+dark="[^"]+"\s+alt="([^"]+)"', open(path).read()):
        alts[m.group(1)] = m.group(2)
names = sorted(n for n in alts if glob.glob(f"fit/{n}.png"))
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").eval()
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
with torch.no_grad():
    def image(folder):               # each tower's pooled output, then CLIP's projection into the shared space
        px = proc(images=[Image.open(f"{folder}/{n}.png").convert("RGB") for n in names], return_tensors="pt")
        return model.visual_projection(model.vision_model(pixel_values=px["pixel_values"]).pooler_output)
    def text(captions):              # CLIP reads at most 77 tokens; longer alt texts are cut
        enc = proc(text=captions, return_tensors="pt", padding=True, truncation=True)
        out = model.text_model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
        return model.text_projection(out.pooler_output)
    caps = {"the post's alt text": [alts[n] for n in names],
            "the file name as words": [n.replace("-", " ") for n in names]}
    print(f"{len(names)} diagrams; chance of the right match at random: 1 in {len(names)} ({1 / len(names):.0%})\n")
    print(f"{'diagrams':20s} {'caption':24s} {'caption -> diagram':>19s} {'diagram -> caption':>19s}  (right one first)")
    for folder, label in (("fit", "as drawn"), ("notext", "with labels hidden")):
        img = image(folder)
        for name, c in caps.items():
            S = torch.nn.functional.normalize(img, dim=-1) @ torch.nn.functional.normalize(text(c), dim=-1).T
            i2t = int((S.argmax(1) == torch.arange(len(names))).sum())
            t2i = int((S.argmax(0) == torch.arange(len(names))).sum())
            print(f"{label:20s} {name:24s} {t2i:>12d} of {len(names)} {i2t:>12d} of {len(names)}")
    n_tok = [len(proc.tokenizer(a).input_ids) for a in caps["the post's alt text"]]
    print(f"\nalt texts: median {sorted(n_tok)[len(n_tok) // 2]} tokens, {sum(x > 77 for x in n_tok)} of {len(n_tok)} cut at 77")

The post printed:

```text
19 diagrams; chance of the right match at random: 1 in 19 (5%)

diagrams             caption                   caption -> diagram  diagram -> caption  (right one first)
as drawn             the post's alt text                 5 of 19            4 of 19
as drawn             the file name as words              4 of 19            3 of 19
with labels hidden   the post's alt text                 1 of 19            2 of 19
with labels hidden   the file name as words              1 of 19            2 of 19

alt texts: median 40 tokens, 3 of 19 cut at 77
```